# Download SAMBBA Data from the CEDA Archive

## Purpose

This notebook automates the download of all FAAM aircraft datasets used in this study from the CEDA archive. Authentication is performed using CEDA's X.509 certificate service, after which the selected SAMBBA flight directories are downloaded to a local project directory.

## Workflow

1. Install the required authentication package.
2. Import the Python libraries used throughout the notebook.
3. Configure certificate locations and CEDA authentication services.
4. Verify or generate valid X.509 credentials.
5. Define the local download directory.
6. Specify the list of SAMBBA flight directories to retrieve.
7. Download all files using `wget` with authenticated access.

## Inputs

- Valid CEDA user account.
- Access to the FAAM SAMBBA archive.
- List of SAMBBA flight identifiers.

## Outputs

- Local copy of the selected FAAM folders and files stored in the project data directory.


In [ ]:
%pip install ContrailOnlineCAClient

In [1]:
# Import standard libraries
import os
import datetime
import requests
# Import third-party libraries
from cryptography import x509
from cryptography.hazmat.backends import default_backend
from contrail.security.onlineca.client import OnlineCaClient
import subprocess

In [2]:
CERTS_DIR = os.path.expanduser('~/.certs')
if not os.path.isdir(CERTS_DIR):
    os.makedirs(CERTS_DIR)

TRUSTROOTS_DIR = os.path.join(CERTS_DIR, 'ca-trustroots')
CREDENTIALS_FILE_PATH = os.path.join(CERTS_DIR, 'credentials.pem')

TRUSTROOTS_SERVICE = 'https://slcs.ceda.ac.uk/onlineca/trustroots/'
CERT_SERVICE = 'https://slcs.ceda.ac.uk/onlineca/certificate/'

In [3]:
def cert_is_valid(cert_file, min_lifetime=0):
    """
    Returns boolean - True if the certificate is in date.
    Optional argument min_lifetime is the number of seconds
    which must remain.

    :param cert_file: certificate file path.
    :param min_lifetime: minimum lifetime (seconds)
    :return: boolean
    """
    try:
        with open(cert_file, 'rb') as f:
            crt_data = f.read()
    except IOError:
        return False

    try:
        cert = x509.load_pem_x509_certificate(crt_data, default_backend())
    except ValueError:
        return False

    now = datetime.datetime.now()
    return (cert.not_valid_before <= now
            and cert.not_valid_after > now + datetime.timedelta(0, min_lifetime))

In [ ]:
def setup_credentials():
    """
    Download and create required credentials files.

    Return True if credentials were set up.
    Return False if credentials were already set up.
    """
    if cert_is_valid(CREDENTIALS_FILE_PATH):
        print('[INFO] Security credentials already set up.')
        return False

    username = 'ceda_username'  # Replace with your CEDA username
    password = 'ceda_password'  # Replace with your CEDA password

    if not username or not password:
        raise ValueError("CEDA_USERNAME and CEDA_PASSWORD environment variables are required")

    onlineca_client = OnlineCaClient()
    onlineca_client.ca_cert_dir = TRUSTROOTS_DIR

    trustroots = onlineca_client.get_trustroots(
        TRUSTROOTS_SERVICE,
        bootstrap=True,
        write_to_ca_cert_dir=True)

    key_pair, certs = onlineca_client.get_certificate(
        username,
        password,
        CERT_SERVICE,
        pem_out_filepath=CREDENTIALS_FILE_PATH)

    print('[INFO] Security credentials set up.')
    return True

In [5]:
# Define the URL and output directory
URL = "https://dap.ceda.ac.uk/badc/faam/data/2012/b731-sep-14/"
OUTPUT_DIR = '/home/jovyan/SAMBBA_data/'  # Change this to your desired directory

In [6]:
# Base URL and output directory
BASE_URL = "https://dap.ceda.ac.uk/badc/faam/data/2012/"

In [16]:
# List of URL suffixes
url_suffixes = [
    "b729-aug-28", "b730-sep-04", "b731-sep-14", "b732-sep-15", "b733-sep-16", 
    "b734-sep-18", "b735-sep-19", "b736-sep-19", "b737-sep-20", "b738-sep-22", 
    "b739-sep-23", "b740-sep-25", "b741-sep-26", "b742-sep-27", "b743-sep-27", 
    "b744-sep-28", "b745-sep-28", "b746-sep-29", "b747-oct-01", "b748-oct-02", 
    "b749-oct-03", "b750-oct-03"
]

In [17]:
# Define a function to download files using wget and X.509 credentials
def download_data_with_wget():
    """
    Download files from a list of URLs using wget with X.509 authentication.

    :param url_list: List of URLs to download.
    :param output_dir: Directory where the files should be saved.
    """
    # Ensure credentials are set up
    try:
        setup_credentials()  # Ensure the certificate is valid
    except ValueError as e:
        print(f"[ERROR] Credential setup failed: {e}")
        return

    for suffix in url_suffixes:
        url = BASE_URL + suffix + "/"
        command = [
            "wget",
             "-e", "robots=off",   
            "--certificate", CREDENTIALS_FILE_PATH,  # Use X.509 certificate for authentication
            "--private-key", CREDENTIALS_FILE_PATH,  # Use the same file as private key (usually works for CEDA)
            "--ca-directory", TRUSTROOTS_DIR,  # Directory for trusted CA certificates
            "--no-check-certificate",  # Disable SSL verification if necessary (can be removed for security)
            "--mirror",  # Enable full mirroring
            "--no-parent",  # Prevent going up directories
            "-r",
            "-P", OUTPUT_DIR,  # Specify output directory
            url  # Target URL
        ]
        
        try:
            subprocess.run(command, check=True)
            print(f"Download completed successfully for {suffix} and saved to {OUTPUT_DIR}")
        except subprocess.CalledProcessError as e:
            print(f"Error during download of {suffix}: {e}")

In [ ]:
download_data_with_wget()